# Task 2B: peak-day allocation

Run from `Datathon/` after restoring the local competition datasets and checker copies. Copy these cells into the shared final notebook. The last cell also demonstrates Task 1/2A inference; run their pipelines first to create local models. Task 2B uses no trained model. Source data and notebook outputs are kept local.


In [ ]:
from pathlib import Path
import json
import subprocess
import sys
import pandas as pd
ROOT = Path.cwd()
assert (ROOT / 'task2b/common.py').exists(), 'Start this notebook from Datathon/'
sys.path.insert(0, str(ROOT))
if hasattr(sys.stdout, 'reconfigure'):
    sys.stdout.reconfigure(encoding='utf-8')
from task2b.common import load_orders, load_fleet, SUBMISSION, REPORT_DIR
from task2b.policy import protected
orders, fleet = load_orders(), load_fleet()
print('Orders:', len(orders), 'Available vehicles:', int(fleet.available.sum()))
print(orders.groupby(['brand', 'temp_requirement'])[['order_volume_m3', 'order_weight_kg']].sum())


## Capacity and policy

Whole orders, one brand and district per trip, compatible depot/access/temperature, two trips total, 270 Fresh minutes and 480 combined Style/Tech minutes. Rank protected order count, chilled volume, total orders, outlets, then fewer minutes. Flags are order-level; protect either flag once. S1-078 cannot fit an available vehicle.


In [ ]:
subprocess.run([sys.executable, 'task2b/analysis.py'], check=True)
print('Protected orders:', sum(protected(r) for r in orders.itertuples()))


## Reproduce all Task 2B phases

This runs greedy, the exact solver-free chilled search, independent exhaustive tests, window diagnostics, repaired-reefer sensitivity, the policy and submission. Sensitivity can take a few minutes. Optimal minutes are conditional on fixed ambient packing.


In [ ]:
subprocess.run([sys.executable, 'task2b/run_all.py'], check=True)
summary = json.loads((REPORT_DIR / 'optimization_summary.json').read_text(encoding='utf-8'))
print('Greedy:', summary['greedy_score'])
print('Exact:', summary['optimized']['full_score'])
print('Gain in chilled m3:', summary['chilled_volume_gain_m3'])
print('Priority volume cost m3:', summary['priority_volume_cost_m3'])


## Audits and written policy

Window waiting and approximate fuel are diagnostics. The no-return simulation does not certify physical execution or weekly fuel compliance. Raw arrival after close defines lateness; late unloading completion is reported separately. Individual chilled deferrals are joint choices, while the Style oversized order is individually impossible.


In [ ]:
print((REPORT_DIR / 'audit_report.txt').read_text(encoding='utf-8'))
print((REPORT_DIR / 'sensitivity_summary.json').read_text(encoding='utf-8'))
print(Path('docs/handover/TASK2B_POLICY.md').read_text(encoding='utf-8'))


## Final cell: saved-model inference and allocation

Loads existing Task 1 and Task 2A models, prints inputs and predictions, then displays/checks Task 2B. No training occurs in this cell.


In [ ]:
import runpy
namespace = runpy.run_path('docs/handover/inference_cell.py')
for task in ('task1', 'task2a', 'task2b'):
    saved = pd.read_csv(f'outputs/submission_{task}.csv', float_precision='round_trip')
    assert namespace[f'{task}_pred'].equals(saved), task
print('All three submissions match the final cell: PASSED')
